# 개인 메쉬 (SMPL + D) 만들기
돌기 세션(정지 프레임)의 실루엣 + LiDAR depth 로 **체형 β (공유) + 프레임별 자세 → 옷·신발 표면 D** 를 맞춥니다.
- 입력: NB1~NB4 를 돌린 세션 (예: `0729_stand_turn`)
- 체형 β: STAND + T자세 전부 / 옷·신발 D: STAND 만 / 검증: STAND 정지 구간마다 1장씩 뺀 9장
- 출력: `sessions/{SESSION}/07_personal/` (그림, 보고서) + 통과 시 `assets/personal_mesh_turn.npz`
- 런타임: GPU 권장 (T4 로 10~20분)

In [ ]:
#@title 설정
ROOT    = "/content/drive/MyDrive/walking_0729"   #@param {type:"string"}
SESSION = "0729_stand_turn"                        #@param {type:"string"}
ITERS_A = 600    #@param {type:"integer"}
ITERS_B = 1500   #@param {type:"integer"}
ITERS_D = 1500   #@param {type:"integer"}
BATCH   = 8      #@param {type:"integer"}
D_CLOTH = 0.05   #@param {type:"number"}
D_SHOE  = 0.04   #@param {type:"number"}
W_LAP   = 300    #@param {type:"number"}
SAVE_AS = "personal_mesh_turn_slim.npz"   #@param {type:"string"}
OUT_NAME = "07_personal_slim"   #@param {type:"string"}
LOOSE_COV_W = 0.0   #@param {type:"number"}
# 0 = 옷 아래 몸 (티셔츠·반바지 부분은 몸이 채우지 않음 → 날씬). 1 = 이전 방식 (옷까지 채움)
D_FID_MIN = 1470   #@param {type:"integer"}
# 옷·신발 D 는 STAND(팔 내림, fid >= 1470) 만으로 — 보행 때 팔 모양과 같음. T자세(556~1432) 는 체형 β 추정에만 사용
VAL_FIDS = "1486,1618,1759,1884,2038,2190,2323,2446,2611"   #@param {type:"string"}
# 검증용: STAND 정지 구간마다 1장씩 피팅에서 빼고, 끝에 맨몸 SMPL vs SMPL+D 를 그 프레임에서 비교

from google.colab import drive
drive.mount("/content/drive")
import os, sys, torch
SESS = f"{ROOT}/sessions/{SESSION}"; OUT = f"{SESS}/{OUT_NAME}"
os.makedirs(OUT, exist_ok=True)
os.environ["WALK_ASSETS"] = f"{ROOT}/assets"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (CPU — 느림)")
for p in ("00_extract/rgb", "01_depth/work/out", "01_depth/work/analysis/seg_pass1.npz", "03_hmr2/hmr2_walking_track.npz"):
    print(("OK  " if os.path.exists(f"{SESS}/{p}") else "없음 ") + p)


In [ ]:
#@title 1. 의존성
!pip -q install smplx


In [ ]:
#@title 2. 피팅 (1단계 β·자세 → 점검 → D)
!cd "{ROOT}/scripts/personal" && WALK_ASSETS="{ROOT}/assets" python fit_personal.py --data "{SESS}" --out "{OUT}" \
   --iters_a {ITERS_A} --iters_b {ITERS_B} --iters_d {ITERS_D} --batch {BATCH} \
   --d_cloth {D_CLOTH} --d_shoe {D_SHOE} --w_lap {W_LAP} --d_fid_min {D_FID_MIN} --val_fids "{VAL_FIDS}" --loose_cov_w {LOOSE_COV_W} 2>&1 | grep -v "UserWarning\|Consider using"


In [ ]:
#@title 3. 그림 (얼굴 모자이크) + 보고서
!cd "{ROOT}/scripts/personal" && WALK_ASSETS="{ROOT}/assets" python render_fit.py "{OUT}" "{SESS}" --noD
!cd "{ROOT}/scripts/personal" && WALK_ASSETS="{ROOT}/assets" python render_fit.py "{OUT}" "{SESS}"
if os.path.exists(f"{OUT}/personal_canonical.obj"):
    !cd "{ROOT}/scripts/personal" && python render_canon.py "{OUT}"
import json
from IPython.display import Image, display
r = json.load(open(f"{OUT}/report.json", encoding="utf-8"))
print("β:", [round(x, 2) for x in r["beta"]], " 키(맨몸):", round(r["canon_height"], 3), " 점검 실패:", r["fail"])
if "D_stats" in r:
    print("D 범위 끝 비율:", round(r["D_at_bound_frac"], 3), " 피부 D 중앙값(cm):", round(r["skin_D_median_cm"], 2))
    print("영역별 D (중앙값/최대, cm):", {k: [round(v[0], 2), round(v[1], 2)] for k, v in r["D_stats"].items()})
for tag in ("SMPL", "SMPL+D"):
    if f"validation_mean_{tag}" in r:
        print(f"[검증 프레임 평균] {tag}:", r[f"validation_mean_{tag}"])
for f in ("fit_overlay.jpg", "fit_overlay_D.jpg", "canonical_views.png"):
    if os.path.exists(f"{OUT}/{f}"):
        print(f); display(Image(f"{OUT}/{f}", width=1100))


In [ ]:
#@title 4. 점검 통과 시 assets 에 저장 (기존 파일은 덮어쓰지 않음)
import shutil, time
src = f"{OUT}/personal_mesh.npz"
if os.path.exists(src) and not r["fail"]:
    dst = f"{ROOT}/assets/{SAVE_AS}"
    if os.path.exists(dst):
        dst = dst.replace(".npz", time.strftime("_%m%d_%H%M.npz"))
    shutil.copy(src, dst); print("저장:", dst)
    print("보행 피팅에 쓰려면 NB_RUN_ALL 의 PERSONAL_MESH =", repr(dst))
else:
    print("저장 안 함 — 점검 실패 또는 D 단계 미실행. 위 보고서 확인")
